# Tier 2 — Qwen3-Embedding-0.6B (Azure GPU)

**BSARD RAG Thesis | RQ1 | T2 Qwen3 Experiments**

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC4as_T4_v3` (16 GB VRAM)
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. **Commit D1–D7 result JSONs locally first** — `--skip-done` reads them from the cloned repo
4. Run cells top to bottom

## What this notebook runs

| Exp | Model | Expected GPU time |
|---|---|---|
| EXP-D10 | Qwen3-Embedding-0.6B (plain) | ~25–35 min |
| EXP-D10i | Qwen3-Embedding-0.6B (instruct prefix) | ~5 min (reuses D10 embeddings) |

D1–D7 results are already committed locally and read from the repo via `--skip-done`.
D7 is always skipped (`--skip-d7`) — it completed locally with me5-large as winner.

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3')
    print(result.stderr)

In [ ]:
# ── Cell 2: Clone / pull GitHub repo ─────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

# Verify D1-D6 result JSONs are present (required for --skip-done)
from pathlib import Path
results_dir = Path(CLONE_DIR) / 'output' / 'results' / 'dense_retrieval'
existing = list(results_dir.glob('dense_*.json')) if results_dir.exists() else []
print(f'\nFound {len(existing)} existing result JSON(s) in repo:')
for f in sorted(existing):
    print(f'  {f.name}')
if len(existing) < 6:
    print('\nWARNING: Expected at least 6 result JSONs (D1-D6). Commit them locally first!')

In [ ]:
# # ── Clean up partial venv before retry ───────────────────────────────────────
# import subprocess
# for path in ['/home/azureuser/venv', '/mnt/venv']:
#     r = subprocess.run(['rm', '-rf', path], capture_output=True, text=True)
#     print(f'Removed {path}:', 'OK' if r.returncode == 0 else r.stderr[:100])

# # Check available space
# df = subprocess.run(['df', '-h', '/tmp', '/home'], capture_output=True, text=True)
# print(df.stdout)


In [ ]:
# ── Cell 3: Create isolated venv and install dependencies ───────────────────
import subprocess, sys, os

VENV_DIR = '/tmp/venv'
VENV_PY  = f'{VENV_DIR}/bin/python'
RQ3_DIR  = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'  # sibling component in the mono-repo

def run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

# Detect NVIDIA driver version to pick the correct torch CUDA wheel.
# nvcc shows toolkit version (can be newer than driver); nvidia-smi shows
# the actual driver version which determines maximum CUDA support.
smi = run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'])
driver_str = smi.stdout.strip().split('\n')[0] if smi.stdout.strip() else '0'
try:
    driver_major = int(driver_str.split('.')[0])
except ValueError:
    driver_major = 0
# Driver < 525 supports at most CUDA 11.x → use cu118
cuda_ver = 'cu118' if driver_major < 525 else 'cu121'
print(f'NVIDIA driver: {driver_str} → torch wheel: {cuda_ver}')

# Create venv (skip if already exists)
if not os.path.exists(VENV_PY):
    print('Creating venv ...', end='', flush=True)
    r = run([sys.executable, '-m', 'venv', VENV_DIR])
    if r.returncode != 0:
        raise RuntimeError(f'venv creation failed:\n{r.stderr[-300:]}')
    print(' OK')
else:
    print(f'Venv already exists at {VENV_DIR}')

def venv_pip(*args):
    return run([VENV_PY, '-m', 'pip', 'install', '-q'] + list(args))

# Upgrade pip/setuptools first
print('  pip upgrade ...', end='', flush=True)
r = venv_pip('--upgrade', 'pip', 'setuptools', 'wheel')
print(' OK' if r.returncode == 0 else f' WARN\n{r.stderr[-200:]}')

# torch (CUDA-matched)
print(f'  torch ({cuda_ver}) ...', end='', flush=True)
r = venv_pip('torch', '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}')
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# Core requirements
print('  requirements.txt ...', end='', flush=True)
r = venv_pip('-r', f'{REPO_DIR}/requirements.txt')
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-300:]}')

# Qwen3 needs transformers >= 4.51.0
print('  transformers>=4.51.0 ...', end='', flush=True)
r = venv_pip('transformers>=4.51.0')
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# Extra deps
for pkg, label in [
    ('azure-storage-blob', 'azure-storage-blob'),
    ('tf-keras',           'tf-keras'),
    ('timm>=0.9.2',        'timm'),
]:
    print(f'  {label} ...', end='', flush=True)
    r = venv_pip(pkg)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# faiss
print('  faiss ...', end='', flush=True)
faiss_pkg = 'faiss-gpu' if cuda_ver == 'cu118' else 'faiss-cpu'
r = venv_pip(faiss_pkg)
if r.returncode != 0:
    r = venv_pip('faiss-cpu')
    faiss_pkg = 'faiss-cpu'
print(f' OK ({faiss_pkg})')

# spaCy French model
print('  spaCy fr_core_news_lg ...', end='', flush=True)
sv_r = run([VENV_PY, '-c', 'import spacy; print(spacy.__version__)'])
sv = sv_r.stdout.strip()
_base = 'https://github.com/explosion/spacy-models/releases/download'
installed = False
for ver in [sv, '.'.join(sv.split('.')[:2]) + '.0']:
    whl = f'fr_core_news_lg-{ver}/fr_core_news_lg-{ver}-py3-none-any.whl'
    r = venv_pip(f'{_base}/{whl}')
    if r.returncode == 0:
        installed = True
        break
if not installed:
    raise RuntimeError('spaCy fr_core_news_lg install failed')
print(' OK')

# bsard_evaluation from RQ3 repo
# bsard_evaluation is the RQ3_Autonomous_Evaluation component, already cloned above.
print('  bsard_evaluation ...', end='', flush=True)
r = venv_pip('-e', RQ3_DIR)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

# Final verification
print('\nVerifying venv...')
check = run([VENV_PY, '-c',
    'import torch, sentence_transformers, transformers, spacy; '
    'print(f"torch {torch.__version__}, cuda: {torch.cuda.is_available()}"); '
    'print(f"transformers {transformers.__version__}"); '
    'spacy.load("fr_core_news_lg"); print("spaCy OK")'
])
print(check.stdout or check.stderr[-300:])
if check.returncode != 0:
    raise RuntimeError('Venv verification failed — check errors above')
print('All dependencies ready. Venv:', VENV_PY)


In [ ]:
# ── Cell 4: Download corpus from Azure Blob ───────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'], check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path

OUTPUT_DIR  = Path(REPO_DIR) / 'output'
EMBED_DIR   = OUTPUT_DIR / 'embeddings'
RESULTS_DIR = OUTPUT_DIR / 'results' / 'dense_retrieval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

downloads = {
    'bsard_articles_dedup.parquet': OUTPUT_DIR,
    'bsard_corpus.db':              OUTPUT_DIR,
}

client = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

for blob_name, dest_dir in downloads.items():
    dest_path = Path(dest_dir) / blob_name
    if dest_path.exists():
        print(f'  Already exists: {blob_name} ({dest_path.stat().st_size / 1e6:.1f} MB)')
        continue
    print(f'  Downloading {blob_name} ...', end='', flush=True)
    with open(dest_path, 'wb') as f:
        client.get_blob_client(blob_name).download_blob().readinto(f)
    print(f' done ({dest_path.stat().st_size / 1e6:.1f} MB)')

print('\nAll data files ready.')

In [ ]:
# ── Cell 5: Pre-flight checks ────────────────────────────────────────────────
import os, subprocess
from pathlib import Path

VENV_PY = '/tmp/venv/bin/python'
os.chdir(REPO_DIR)

checks_passed = True

# Corpus files
for f in ['output/bsard_articles_dedup.parquet', 'output/bsard_corpus.db']:
    p = Path(f)
    if p.exists():
        print(f'  {f}: OK ({p.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'  MISSING: {f}')
        checks_passed = False

# GPU
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'\nGPU: {gpu}')

# transformers version in venv (Qwen3 needs >= 4.51.0)
tv_r = subprocess.run([VENV_PY, '-c',
    'import transformers; from packaging.version import Version; '
    'tv=transformers.__version__; '
    'ok=Version(tv)>=Version("4.51.0"); '
    'print(tv, "OK" if ok else "TOO OLD")'],
    capture_output=True, text=True)
tv_out = tv_r.stdout.strip() or tv_r.stderr.strip()
print(f'transformers (venv): {tv_out}')
if 'TOO OLD' in tv_out or tv_r.returncode != 0:
    checks_passed = False

print(f'\n{"All checks passed." if checks_passed else "FIX ISSUES ABOVE before running Cell 6."}')


In [ ]:
# import subprocess
# VENV_PY = '/tmp/venv/bin/python'

# # Check torch CUDA status
# r = subprocess.run([VENV_PY, '-c',
#     'import torch; '
#     'print("torch:", torch.__version__); '
#     'print("torch.version.cuda:", torch.version.cuda); '
#     'print("cuda available:", torch.cuda.is_available())'],
#     capture_output=True, text=True)
# print(r.stdout or r.stderr[:400])

# # Check CUDA libraries visible to the system
# r2 = subprocess.run(['ldconfig', '-p'], capture_output=True, text=True)
# libcuda = [l for l in r2.stdout.splitlines() if 'libcuda' in l.lower()]
# print('libcuda entries:', libcuda or 'NONE FOUND')


In [ ]:
# import subprocess
# VENV_PY = '/tmp/venv/bin/python'

# print('Reinstalling torch pinned to 2.5.1+cu121 ...')
# r = subprocess.run([VENV_PY, '-m', 'pip', 'install', '-q',
#     'torch==2.5.1+cu121',
#     '--index-url', 'https://download.pytorch.org/whl/cu121'],
#     capture_output=True, text=True)
# print('Exit code:', r.returncode)
# print(r.stdout[-200:] or r.stderr[-200:])

# # Verify
# r2 = subprocess.run([VENV_PY, '-c',
#     'import torch; print(torch.__version__, "| cuda:", torch.cuda.is_available())'],
#     capture_output=True, text=True)
# print(r2.stdout or r2.stderr[:200])


In [ ]:
import subprocess
VENV_PY = '/tmp/venv/bin/python'

r = subprocess.run([VENV_PY, '-m', 'pip', 'install', '-q',
    'torchvision==0.20.1+cu121',
    '--index-url', 'https://download.pytorch.org/whl/cu121'],
    capture_output=True, text=True)
print('Exit code:', r.returncode)
print(r.stdout[-200:] or r.stderr[-200:])


In [ ]:
# ── Cell 6: Run Qwen3 experiments ────────────────────────────────────────────
# --only-qwen3 runs exclusively D10 and D10i — all other models and D7 are skipped.
# Expected total time on T4 GPU: ~30-40 min
import subprocess, os

VENV_PY = VENV_PY = '/tmp/venv/bin/python'
os.chdir(REPO_DIR)

cmd = [
    VENV_PY,
    'scripts/evaluation/tier2/run_dense_experiments.py',
    '--skip-audit',
    '--only-qwen3',
    '--device', 'cuda',
]

print('Command:', ' '.join(cmd))
print('=' * 70)

result = subprocess.run(cmd, cwd=REPO_DIR)
print('=' * 70)
print('Exit code:', result.returncode)
if result.returncode != 0:
    print('ERROR — check output above.')


In [ ]:
# ── Cell 7: Upload Qwen3 embeddings to Azure Blob (for future reuse) ──────────
# Uploads the .npy embedding files so they can be downloaded in future runs
# without re-encoding (saves ~25-35 min per run).
from azure.storage.blob import ContainerClient, BlobClient
from pathlib import Path

EMBED_DIR = Path(REPO_DIR) / 'output' / 'embeddings'
client    = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)

qwen3_files = list(EMBED_DIR.glob('*qwen3*')) + list(EMBED_DIR.glob('*Qwen3*'))

if not qwen3_files:
    print('No Qwen3 embedding files found — skipping upload.')
    print(f'  (looked in {EMBED_DIR})')
else:
    print(f'Uploading {len(qwen3_files)} Qwen3 embedding file(s) to Azure Blob...')
    for path in sorted(qwen3_files):
        blob_name = path.name
        size_mb   = path.stat().st_size / 1e6
        print(f'  {blob_name} ({size_mb:.1f} MB) ...', end='', flush=True)
        blob_client = client.get_blob_client(blob_name)
        with open(path, 'rb') as f:
            blob_client.upload_blob(f, overwrite=True)
        print(' done')
    print('\nUpload complete.')

In [ ]:
# ── Cell 8: Results summary ───────────────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(REPO_DIR) / 'output' / 'results' / 'dense_retrieval'
all_files   = sorted(results_dir.glob('dense_*.json'))

print(f'{"Experiment":<45}  R@10    R@100   MRR@10')
print('-' * 78)
for f in all_files:
    d = json.loads(f.read_text())
    m = d['metrics']
    print(f"  {d['experiment_id']:<43}  {m['Recall@10']:.4f}  "
          f"{m['Recall@100']:.4f}  {m['MRR@10']:.4f}")

In [ ]:
# ── Cell 9: Commit and push results to GitHub ─────────────────────────────────
import os, subprocess
from pathlib import Path

# ── fill these in if Cell 0 wasn't run ────────────────────────────────────────
GITHUB_TOKEN = ''   # your token
REPO         = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root
GIT_NAME     = 'MariusPasch'
GIT_EMAIL    = 'paschalidismarios@gmail.com'
# ──────────────────────────────────────────────────────────────────────────────

def git(args):
    return subprocess.run(['git'] + args, cwd=REPO_DIR, capture_output=True, text=True)

git(['config', 'user.email', GIT_EMAIL])
git(['config', 'user.name',  GIT_NAME])
git(['remote', 'set-url', 'origin', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'])

results_dir = Path(REPO_DIR) / 'output' / 'results' / 'dense_retrieval'
new_files = [
    f'output/results/dense_retrieval/{f.name}'
    for f in results_dir.glob('dense_qwen3_*.json')
]

staged = [f for f in new_files if Path(f'{REPO_DIR}/{f}').exists()]
for f in staged:
    git(['add', '-f', f])   # -f to override .gitignore

print(f'Staged {len(staged)} file(s):')
for f in staged:
    print(f'  {f}')

if not staged:
    print('No new result files found to commit.')
else:
    status = git(['status', '--short']).stdout.strip()
    if status:
        commit = git(['commit', '-m',
            'T2 D10/D10i results — Azure T4 GPU\n\n'
            ''])
        print('Commit:', commit.stdout.strip() or commit.stderr.strip())
    else:
        print('Nothing new to commit — already up to date.')

print('\nPulling remote changes...')
pull = git(['pull', '--rebase', 'origin', 'main'])
print(pull.stdout.strip() or pull.stderr.strip())
if pull.returncode != 0:
    print('Pull failed — resolve conflicts manually.')
else:
    push = git(['push', 'origin', 'main'])
    if push.returncode == 0:
        print('Pushed.')
    else:
        print(f'Push failed: {push.stderr[-400:]}')
